# HW4: Evaluating a Classifier, and Writing the Rule

**Foundations of Data Science. Posted chapter 8, due before chapter 10. 100 points.**

Use your project classifier if you have one; otherwise the NHANES diabetes model with a **different feature set from the lecture**: drop `waist_cm` and `dbp`, and add `income_ratio` as a categorical band (below poverty, 1-2x, 2x+). The questions assume the NHANES version.

**Rules.** Runs top to bottom from `data/processed/nhanes.parquet`. Stratified 60/20/20 split with `model_tuner`, `random_state=42`. Preprocessing inside the pipeline. The test set is touched only in Q3 onward and only once per quantity. Written answers in the **Answer** cells.

## Setup

<!-- nb-links -->
[Download this notebook](hw4_classifier_evaluation.ipynb?download=1){download="hw4_classifier_evaluation.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/assignments/chapter_08/hw4_classifier_evaluation.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course_students"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from model_tuner import Model

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})
adults["income_band"] = pd.cut(adults["income_ratio"], bins=[0, 1, 2, 5.01], labels=["below poverty", "1-2x", "2x+"], right=False).astype(str)

num = ["age", "bmi", "sbp"]
cat = ["sex", "race_eth", "income_band"]
X, y = adults[num + cat], adults["diabetes"]
print(X.shape, " base rate:", y.mean().round(3))

## Q1. Fit (10 pts)

Logistic regression through `model_tuner.Model`, C chosen on validation from `[0.01, 0.1, 1, 10]`. Report the chosen C and the base rate in each split.

In [ ]:
# Q1 code


**Answer (Q1):**



## Q2. The cutoff (20 pts)

On the **validation** set, find the cutoff that gives 80% sensitivity (use `find_optimal_threshold_beta` or a sweep). Also compute Youden's J cutoff. **Answer**: both cutoffs, and which you'd pick for a screening program and why.

In [ ]:
# Q2 code


**Answer (Q2):**



## Q3. Test-set metrics at the chosen cutoff (20 pts)

Confusion matrix via `show_confusion_matrix` with the classification report. Then a table: sensitivity, specificity, PPV, NPV, share flagged. **Answer**: the four rates in words a clinician would use ("of the people who have it, we catch...").

In [ ]:
# Q3 code


**Answer (Q3):**



## Q4. Curves and calibration (15 pts)

ROC and PR curves (`show_roc_curve`, `show_pr_curve`) and a calibration curve with Brier score. **Answer**: AUC and AP, whether the model is calibrated in the range where most patients sit, and what the PR curve's baseline is and why it matters here.

In [ ]:
# Q4 code


**Answer (Q4):**



## Q5. Intervals (15 pts)

`evaluate_bootstrap_metrics` with `ci_method="percentile"`, 1,000 resamples, at your cutoff, for AUC, sensitivity, precision, specificity. **Answer**: the four with intervals, and one sentence on what the default `ci_method="t"` would have given you instead.

In [ ]:
# Q5 code


**Answer (Q5):**



## Q6. The rule (20 pts)

A table for cutoffs 0.10, 0.14, 0.20, 0.30: share flagged, share of cases caught, tests per case found, cases missed per 1,000 adults. Then write the recommendation **in the decision-maker's language**, four sentences maximum, including what the clinic gives up at the cutoff you recommend. Compare in one sentence to the lecture model (which had waist and dbp): is this feature set meaningfully worse?

In [ ]:
# Q6 code


**Answer (Q6):**

